In [1]:
import os
import warnings
import langchain_core
warnings.filterwarnings("ignore")

# Ensure Working Directory is set to Project Root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(f"Working Directory set to: {os.getcwd()}")

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Initialize LLM & AgentTracer
from app.utils.llm import get_llm
from modules.common.agent_tracer import AgentTracer
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

llm = get_llm("openai:gpt-4o-mini", temperature=0.0)
tracer = AgentTracer(log_dir="./artifacts/logs", verbose=True)
print("Setup completed successfully!")

Working Directory set to: /mnt/c/Users/hyoun/Desktop/github/frontier-agent-lab
Setup completed successfully!


# Hermes Agent: 4계층 메모리 아키텍처 및 Closed Learning Loop 실습

본 노트북은 **Hermes Agent**의 혁신적인 **4계층 메모리 구조(4-Layer Memory Architecture)**와 **Closed Learning Loop(자가 학습 순환 구조)**를 `frontier-agent-lab` 환경에서 직접 실습하고 검증하기 위해 제작되었습니다.

---

### 🎯 학습 목표
1. **L1 ~ L4 메모리 계층 구조 이해**: Short-term Working Memory부터 Long-term Episodic/Semantic Memory 및 Procedural Memory까지의 분리와 역할을 이해합니다.
2. **L3 Semantic Memory 스토어 제어**: `§` 구분자 기반의 `MEMORY.md` / `USER.md` 스토리지, **Frozen Snapshot** 패턴, 용량 제한(Capacity Bounding) 동작을 확인합니다.
3. **L2 Episodic Memory & Lineage 인출**: SQLite FTS5 기반의 세션 요약 검색과 메시지 **Anchor 선정 + Bookend (첫 3개 + tail 3개) Lineage 인출** 메커니즘을 경험합니다.
4. **4-Layer PromptAssembler 및 캐싱 경계**: `__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__` 기준 정적/동적 프롬프트 분리와 `recalled_memory` 동적 주입을 확인합니다.
5. **Closed Learning Loop & Background Review**: `after_agent` 미들웨어 훅에서 비동기 데몬 스레드로 LLM이 대화를 리뷰하여 장기 메모리를 자동 갱신하는 과정을 검증합니다.

## 🏗️ 4계층 메모리 & Closed Learning Loop 전체 구조

Hermes 에이전트는 대화 흐름을 방해하지 않으면서 턴(Turn) 진행 중 및 세션 완료 후 메모리를 추출하여 장기 기억으로 전환합니다.

![Hermes Agent 4-Layer Memory Architecture](assets/hermes_memory_architecture.svg)


## 1. L3 Semantic Memory (의미론적 장기 기억스토어)

**L3 Semantic Memory**는 에이전트가 지속적으로 보유해야 하는 **세상에 대한 팩트(`MEMORY.md`)**와 **사용자에 대한 프로필/선호도(`USER.md`)**를 보관하는 장기 기억 장치입니다.

--- 

### 🛠️ Semantic Memory CRUD Actions (`memory` 도구 동작 메커니즘)

L3 메모리 스토어는 용량 무한 증식을 방지하고 기억의 질을 높이기 위해 **3가지 액션(Action)**을 지원하며, LLM이 백그라운드 리뷰 시 이를 **자율적으로 판단하여 격발**합니다.

| Action | 설명 및 사용 시점 | 저장/동작 방식 |
|---|---|---|
| **`add`** | 기존 기억에 없는 **완전히 새로운 독립적 팩트/선호도**가 등장했을 때 사용 | `§` (Section Sign) 구분자로 독립된 엔트리 추가 |
| **`replace`** | 기존 팩트와 연관되거나 업그레이드된 정보가 등장했을 때 **스마트 병합(Merge & Compact)** | 기존 `old_text`를 찾아 문장을 한 덩어리로 압축·수정 (`§` 개수 유지) |
| **`remove`** | 기존 팩트가 폐기되거나 더 이상 유효하지 않게 되었을 때 **기억 삭제** | 대상 엔트리를 찾아 스토어에서 완전 제거 |

--- 

### 1.1 L3 Semantic Memory의 생성 (Generation & Storage)

L1 Working Memory(대화 맥락)에서 L3 Semantic Memory로 팩트가 생성되어 기록되는 통로는 **2가지**가 있습니다.

1. **에이전트 자율 생성 (`memory` 도구 직접 호출)**:
   - 대화 진행 중 에이전트가 중요한 정보를 전달받으면 자율적으로 `@tool memory(action='add', target='memory', content='...')` 도구를 격발하여 저장합니다.

2. **Closed Learning Loop 자동 생성 (`MemoryMiddleware` 백그라운드 학습)**:
   - 턴이 끝난 후 `after_agent` 미들웨어 훅에서 비동기 데몬 스레드가 작동합니다.
   - LLM이 최근 L1 대화 내역을 관찰하여 신규 팩트나 유저 선호도를 자동으로 추출(Fact Extraction)하고 XML 태그(`<call:memory ... />`) 또는 함수 형태로 파싱하여 파일에 저장합니다.

아래 실습 코드에서 두 가지 생성 방식을 모두 시연합니다.

In [2]:
import tempfile
import os
from modules.hermes.memory_store import SemanticMemoryStore
from modules.hermes.memory_middleware import MemoryMiddleware
from modules.hermes.session_store import EpisodicStore
from app.utils import get_llm

print("=== [1.1 L3 Semantic Memory 생성 실습] ===\n")

# 1. 실습용 임시 디렉토리 생성 및 스토어 초기화
tmp_dir = tempfile.mkdtemp()
semantic_store = SemanticMemoryStore(memory_dir=tmp_dir)
semantic_store.load_from_disk() # MEMORY.md와 USER.md를 읽습니다. 현재는 비어있습니다.

# -----------------------------------------------------
# [방법 1] 에이전트 도구를 통한 직접 생성 (Direct Tool Call)
# -----------------------------------------------------
print("📌 [방법 1] 스토어 add() 메서드를 통한 팩트 직접 저장")
res1 = semantic_store.add("memory", "Project runs on Python 3.12 + LangChain 0.3.")
res2 = semantic_store.add("user", "User prefers Korean responses with technical terms in English.")
print(f"  - MEMORY.md 저장 결과: {res1['message']} (사용량: {res1['usage']})")
print(f"  - USER.md 저장 결과:   {res2['message']} (사용량: {res2['usage']})\n")

print("\n📂 [1차로 생성된 MEMORY.md 파일 내용]")
with open(os.path.join(tmp_dir, "MEMORY.md"), "r", encoding="utf-8") as f:
    print(f.read())

print("📂 [최종 생성된 USER.md 파일 내용]")
with open(os.path.join(tmp_dir, "USER.md"), "r", encoding="utf-8") as f:
    print(f.read())

=== [1.1 L3 Semantic Memory 생성 실습] ===

📌 [방법 1] 스토어 add() 메서드를 통한 팩트 직접 저장
  - MEMORY.md 저장 결과: Entry added. (사용량: 2% — 44/2,200 chars)
  - USER.md 저장 결과:   Entry added. (사용량: 4% — 62/1,375 chars)


📂 [1차로 생성된 MEMORY.md 파일 내용]
Project runs on Python 3.12 + LangChain 0.3.
📂 [최종 생성된 USER.md 파일 내용]
User prefers Korean responses with technical terms in English.


In [3]:
# -----------------------------------------------------
# [방법 2] Closed Learning Loop를 통한 L1 대화 자동 팩트 추출
# -----------------------------------------------------
print("📌 [방법 2] L1 대화 내역으로부터 LLM 백그라운드 자동 팩트 추출")
episodic_store = EpisodicStore(os.path.join(tmp_dir, "episodic.db"))
middleware = MemoryMiddleware(semantic_store=semantic_store, episodic_store=episodic_store, review_llm=llm)

# 유저가 새로운 정보(테스트 프레임워크 선호도)를 밝힌 L1 대화 샘플
l1_conversation = [
    {"role": "user", "content": "앞으로 파이썬 코드를 작성할 때는 단위 테스트로 zawsze pytest를 써줘."},
    {"role": "assistant", "content": "네, 알겠습니다! pytest를 기본 테스트 프레임워크로 기억하겠습니다."}
]

# 백그라운드 리뷰 실행 (L1 대화 리뷰 -> LLM이 팩트 추출 -> memory() 자동 실행)
middleware._review_semantic_memory(l1_conversation)

# -----------------------------------------------------
# [결과 확인] 생성된 디스크 파일(MEMORY.md / USER.md)의 내용 조회
# -----------------------------------------------------
print("\n📂 [최종 생성된 MEMORY.md 파일 내용]")
with open(os.path.join(tmp_dir, "MEMORY.md"), "r", encoding="utf-8") as f:
    print(f.read())

print("📂 [최종 생성된 USER.md 파일 내용]")
with open(os.path.join(tmp_dir, "USER.md"), "r", encoding="utf-8") as f:
    print(f.read())

📌 [방법 2] L1 대화 내역으로부터 LLM 백그라운드 자동 팩트 추출

📂 [최종 생성된 MEMORY.md 파일 내용]
Project runs on Python 3.12 + LangChain 0.3; user prefers pytest for unit testing.
📂 [최종 생성된 USER.md 파일 내용]
User prefers Korean responses with technical terms in English.
§
User prefers to use pytest for unit testing in Python code.


### 🤖 LLM은 어떻게 `add` 대신 `replace` 병합을 자율 판단할까?

`MemoryMiddleware`는 백그라운드 리뷰 시 LLM에게 현재 `MEMORY.md` 및 `USER.md`에 저장된 최신 팩트 목록을 프롬프트로 보여줍니다.
LLM은 지침(`Merge overlapping entries using memory(action="replace", ...)`)에 따라 다음과 같이 자율 판단합니다:

- **상황 예시**: `MEMORY.md`에 이미 `Project runs on Python 3.12 + LangChain 0.3.`이 있는 상태에서 유저가 *"pytest를 써줘"*라고 대화함.
- **LLM의 스마트 판단**: 단순 `add`를 남발하면 메모리가 지저분하게 파편화되므로, LLM이 `replace` 액션을 선택하여 `old_text="Project runs on Python 3.12"`를 `"Project runs on Python 3.12 + LangChain 0.3; user prefers pytest for unit testing."` 처럼 **세미콜론(`;`)으로 한 엔트리에 예쁘게 병합 요약**합니다!
- **결과적인 이점**: 독립된 팩트는 `§` 구분자로 늘어나고, 연관된 팩트는 세미콜론 `;`으로 압축 병합되어 **제한된 용량(`memory_char_limit: 2200`)을 효율적으로 활용**하게 됩니다.

### 1.2 L3 Semantic Memory의 인출 (Retrieval & Prompt/Tool Injection)

생성된 L3 메모리는 **2가지 방식**으로 인출되어 사용됩니다.

1. **시스템 프롬프트 자동 인출 & Frozen Snapshot 패턴**:
   - 세션이 시작될 때 `load_from_disk()`가 디스크의 마크다운 팩트를 읽어 **Frozen Snapshot(프롬프트 주입용 스냅샷)** 을 만듭니다.
   - 대화 진행 중 메모리가 새로 추가되어도 당해 세션 동안은 시스템 프롬프트가 변경되지 않고 **Frozen Snapshot**이 유지되어 **LLM Prefix Cache(프리픽스 캐시)** 비용 절감 효과를 제공합니다.
   - 새 팩트는 **다음 세션 시작 시 재로드되어 시스템 프롬프트에 자동으로 주입**됩니다.

2. **도구(Tool)를 통한 동적 조회 및 관리**:
   - 필요 시 에이전트는 `memory(action='replace', ...)` 또는 `memory(action='remove', ...)` 도구를 사용하여 기존 기억을 교체하거나 삭제할 수 있습니다.

아래 실습 코드에서 세션 간 Frozen Snapshot 인출 전이와 도구 기반 기억 수정을 확인해보세요.

[1] 시작 시점의 시스템 프롬프트 주입 스냅샷 확인

위 두 번의 실행으로 MEMORY.md와 USER.md의 내용이 생성됐지만, 이번 세션에서는 변경 사항이 프롬프트 캐싱을 위해 시스템 프롬프트에 영향을 주지 않습니다.
어차피 해당 정보는 대화 컨텍스트에 이미 있기 때문입니다.

In [4]:
print("📌 [세션 1 시작 시점] 시스템 프롬프트에 주입될 Frozen Snapshot:")
prompt_snapshot_session1 = semantic_store.format_for_prompt("memory")
print(prompt_snapshot_session1)
print("\n💡 설명: 세션 1이 시작할 당시의 스냅샷이 유지되므로, 세션 도중 새로 추가된 팩트는 세션 1의 시스템 프롬프트에 영향을 주지 않습니다. (Prefix Cache 보존!)\n")

📌 [세션 1 시작 시점] 시스템 프롬프트에 주입될 Frozen Snapshot:
None

💡 설명: 세션 1이 시작할 당시의 스냅샷이 유지되므로, 세션 도중 새로 추가된 팩트는 세션 1의 시스템 프롬프트에 영향을 주지 않습니다. (Prefix Cache 보존!)



### [2] 새로운 세션을 시작할 때 로드 -> 프롬프트 주입 스냅샷 전이 확인

새로운 세션이 시작됐다고 가정합니다. 이때 아래와 같이 문서를 읽게 되면, 에이전트가 기억을 떠올리는 겁니다.

In [5]:
print("📌 [세션 2 시작 시점] 새로운 세션 시작 후 load_from_disk() 실행 결과:")
session2_store = SemanticMemoryStore(memory_dir=tmp_dir)
session2_store.load_from_disk() # 여기서 MEMORY.md와 USER.md를 읽습니다.

📌 [세션 2 시작 시점] 새로운 세션 시작 후 load_from_disk() 실행 결과:


이전 세션에서 저장되었던 모든 팩트가 세션 2의 시스템 프롬프트 Layer 4에 정상적으로 자동 인출되어 주입되었습니다!

In [6]:
prompt_snapshot_session2 = session2_store.format_for_prompt("memory")
print(prompt_snapshot_session2)

══════════════════════════════════════════════
MEMORY (agent's personal notes) [3% — 82/2,200 chars]
══════════════════════════════════════════════
Project runs on Python 3.12 + LangChain 0.3; user prefers pytest for unit testing.


### [3] 도구를 통한 인출 및 기억 수정/삭제 (Replace / Remove)

에이전트는 도구를 통해 기억을 변경할 수도 있습니다. 에이전트가 시간이 지나 바뀐 기술 정보(버전 업그레이드 등)를 능동적으로 수정(replace)하는 과정을 보여줍니다.

In [7]:
print("📌 [도구 활용] 기존 팩트의 수정(Replace) 및 삭제(Remove)")
# 팩트 수정: Python 3.12 -> Python 3.12.3 with LangChain 0.3.5
replace_res = session2_store.replace(
    target="memory",
    old_text="Python 3.12",
    new_content="Project runs on Python 3.12.3 with LangChain 0.3.5; user prefers pytest for unit testing."
)
print(f"  - Replace 수정 결과: {replace_res['message']}")

📌 [도구 활용] 기존 팩트의 수정(Replace) 및 삭제(Remove)
  - Replace 수정 결과: Entry replaced.


In [8]:
print("\n📂 [수정 후 최신 MEMORY.md 파일 내용]")
with open(os.path.join(tmp_dir, "MEMORY.md"), "r", encoding="utf-8") as f:
    print(f.read())


📂 [수정 후 최신 MEMORY.md 파일 내용]
Project runs on Python 3.12.3 with LangChain 0.3.5; user prefers pytest for unit testing.


## 2. L2 Episodic Memory (세션 DB & Lineage 기반 Anchor 인출)

**L2 Episodic Memory**는 과거 세션 대화 전체를 SQLite(`episodic.db`)에 영구 보관하고 검색하는 에피소드 메모리입니다.

### 🔑 핵심 메커니즘
1. **English Summary + FTS5 Discovery**: 세션이 종료될 때 대화 내용을 바탕으로 **영어 요약(Summary)**과 **키워드(Keywords)**를 추출하여 FTS5 전문 검색 테이블(`sessions_fts`)에 인덱싱합니다.
2. **Anchor Selection & Lineage Retrieval**: 과거 세션을 상세 조회할 때 유저 쿼리와 가장 관련성이 높은 메시지를 **Anchor**로 지정하고, 앵커 전후 ±`window` 메시지를 인출합니다.
3. **Bookends (첫 3개 + 끝 3개)**: 세션의 시작 맥락과 최종 결론 맥락을 잃지 않도록 세션의 첫 3개 메시지 및 마지막 3개 메시지를 Bookend로 항상 포함하여 조합합니다.

아래 실습 코드에서는 **5개의 서로 다른 주제 세션(마케팅, 데이터 분석, DevOps, AI/ML 연구, 금융 결제)**을 구축하고 FTS5 검색과 Anchor 인출을 시연한 뒤 `episodic.db`를 리셋 초기화합니다.

In [9]:
import os
import json
import asyncio
from modules.hermes.session_store import EpisodicStore
# 1. artifacts/chat 폴더에서 10개 시나리오 대화 JSON 파일 자동 로드
chat_dir = "./artifacts/chat"
scenario_files = sorted([f for f in os.listdir(chat_dir) if f.endswith(".json")])
scenarios = []
for file_name in scenario_files:
    file_path = os.path.join(chat_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        scenarios.append(json.load(f))
print(f"📦 [artifacts/chat 디렉토리] 총 {len(scenarios)}개 대형 시나리오 대화 세션 로드 완료!\n")

📦 [artifacts/chat 디렉토리] 총 10개 대형 시나리오 대화 세션 로드 완료!



In [10]:
# 2. EpisodicStore 초기화 및 파이널라이즈 실습
async def test_episodic_with_artifacts():
    db_path = os.path.join(tmp_dir, "episodic_demo.db")
    episodic_store = EpisodicStore(db_path=db_path)
    await episodic_store.setup()
    print("=== [1. 10개 대형 세션 파이널라이즈 및 FTS5 인덱싱] ===")
    for sc in scenarios:
        sid = sc["session_id"]
        msgs = sc["messages"]
        summary = await episodic_store.finalize_session(sid, msgs, llm=llm)
        print(f"  - [{sid}] 요약 저장 완료: {summary[:75]}...")


        
    print("\n=== [2. FTS5 전문 검색을 통한 세션 탐색 (Session Discovery)] ===")
    queries = [
        ("marketing campaign server S3 CloudFront Slack", "scenario_01"),
        ("Oracle DB query customer_pii SHA-256 dashboard", "scenario_02"),
        ("K8s cluster HashiCorp Vault secrets retention", "scenario_03"),
        ("GPU fine-tuning PyTorch Weights & Biases DeepSpeed", "scenario_04"),
        ("payment gateway OAuth PCI-DSS audit log", "scenario_05"),
        ("E-Commerce Redis cart TTL Mutex Lock Kafka", "scenario_06"),
        ("Embedded IoT firmware MQTT ECDSA Dual Bank", "scenario_07"),
        ("Game server matchmaking MMR UDP KCP 60Hz", "scenario_08"),
        ("Healthcare DICOM PACS HIPAA AES-256 Lossless", "scenario_09"),
        ("Blockchain Ethereum Solidity The Graph Subgraph", "scenario_10"),
    ]
    matched_count = 0
    for q, expected_sid in queries:
        results = await episodic_store.search_sessions(query=q, top_k=1)
        if results:
            top_match = results[0]
            matched_sid = top_match["session_id"]
            if matched_sid == expected_sid:
                matched_count += 1
                print(f"🔎 쿼리: '{q[:40]}...' -> ✅ MATCHED! [{matched_sid}]")
            else:
                print(f"🔎 쿼리: '{q[:40]}...' -> ⚠️ MISMATCH [{matched_sid}] (Expected: {expected_sid})")
    print(f"\n📊 FTS5 세션 검색 매칭 정확도: {matched_count}/{len(queries)} ({int(matched_count/len(queries)*100)}%) 성공!")


    
    print("\n=== [3. Anchor 키워드 기반 세션 맥락 인출 (Lineage Retrieval)] ===")
    # 대표 3개 세션에서 핵심 키워드 앵커 전후 맥락 및 Bookends(시작 3개 + 끝 3개) 인출
    anchors = [
        ("scenario_01", "CloudFront"),
        ("scenario_06", "Mutex Lock"),
        ("scenario_09", "HIPAA"),
    ]
    for sid, anchor_kw in anchors:
        recalled = await episodic_store.get_anchored_view(sid, anchor_keyword=anchor_kw, window=1)
        print(f"\n📍 [{sid} Anchor '{anchor_kw}' 맥락 인출 결과]: 총 {len(recalled)}개 메시지 (Bookends + Core View)")
        for m in recalled[:3]:
            print(f"  - [{m['role']}] {m['content'][:65]}...")
    await episodic_store.close()
    
    if os.path.exists(db_path):
        os.remove(db_path)
        print("\n🧹 [episodic_demo.db Reset 초기화 완료]")

        
await test_episodic_with_artifacts()

=== [1. 10개 대형 세션 파이널라이즈 및 FTS5 인덱싱] ===
  - [scenario_01] 요약 저장 완료: The conversation involves a marketing team leader discussing the infrastruc...
  - [scenario_02] 요약 저장 완료: The user, a data analyst, discusses various aspects of handling marketing s...
  - [scenario_03] 요약 저장 완료: The user, a DevOps engineer, is establishing a guide for deploying a backen...
  - [scenario_04] 요약 저장 완료: The user, a researcher at an AI lab, is setting up an experimental environm...
  - [scenario_05] 요약 저장 완료: The conversation involves a discussion between a user and a team member abo...
  - [scenario_06] 요약 저장 완료: The conversation involves a team lead discussing the architecture for Redis...
  - [scenario_07] 요약 저장 완료: The conversation involves a user discussing the implementation of an OTA fi...
  - [scenario_08] 요약 저장 완료: The conversation involves a user discussing the design of a real-time multi...
  - [scenario_09] 요약 저장 완료: The conversation involves a user discussing the development of a healthcare

## 3. Prompt Engineering (4-Layer PromptAssembler & Caching Boundary)

Hermes 에이전트의 시스템 프롬프트는 4개의 Layer로 구분되며, **캐싱 경계(Caching Boundary)**를 통해 LLM API 토큰 비용을 최소화합니다.

| Layer | 구체적 내용 | 갱신 주기 & 캐싱 여부 |
|---|---|---|
| **Layer 1** | Static Identity & Fundamental System Rules | 캐싱 대상 (고정) |
| **Layer 2** | Tool Specs (알파벳 정렬) + Skill Index | 캐싱 대상 (고정) |
| `-- BOUNDARY --` | `__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__` | **프리픽스 캐시 경계선** |
| **Layer 3** | Dynamic Session Context (`recalled_memory` 포함) | 매 턴 변경 (동적) |
| **Layer 4** | User & Project Rules (`MEMORY.md`, `USER.md` 스냅샷) | 세션 단위 주입 (동적) |

In [11]:
from modules.hermes.prompt_assembler import PromptAssembler

# 시스템 기본 규칙 (Layer 1)
system_rules = "You are Frontier Agent, an advanced coding assistant equipped with 4-layer memory."

# 샘플 도구 스펙 (Layer 2)
sample_tools = [
    {"name": "web_search", "description": "Search the web for real-time info."},
    {"name": "memory", "description": "Manage long-term semantic memory."},
]

assembler = PromptAssembler(system_rules=system_rules, tool_schemas=sample_tools)

# 동적 세션 컨텍스트 (Layer 3 & 4)
session_context = {
    "cwd": "/mnt/c/Users/hyoun/Desktop/github/frontier-agent-lab",
    "session_id": "demo_session_101",
    "recalled_memory": "[EPISODIC MEMORY]\n- User previously discussed Hermes memory architecture.\n\n[SEMANTIC MEMORY]\n- Project runs on Python 3.12."
}

full_prompt = assembler.build_system_prompt(session_context)
print("=== [Assembled 4-Layer System Prompt Preview] ===\n")
print(full_prompt[:800] + "\n\n... [Truncated] ...")

=== [Assembled 4-Layer System Prompt Preview] ===

=== Layer 1: System Identity & Rules ===
You are Frontier Agent, an advanced coding assistant equipped with 4-layer memory.

=== Layer 2: Tool Capabilities (Alphabetical) ===
### [1] memory
Description: Manage long-term semantic memory.

### [2] web_search
Description: Search the web for real-time info.

__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__

=== Layer 3: Dynamic Session Context ===

Session Information:
- Working Directory: /mnt/c/Users/hyoun/Desktop/github/frontier-agent-lab
- Session ID: demo_session_101
- Host OS: posix

Recalled Memory:
[EPISODIC MEMORY]
- User previously discussed Hermes memory architecture.

[SEMANTIC MEMORY]
- Project runs on Python 3.12.

=== Layer 4: Dynamic Session Documents ===

No dynamic session documents provided.

=== Layer 5: User & Project Rules ===

No proj

... [Truncated] ...


## 4. Memory Middleware & Closed Learning Loop

**MemoryMiddleware**는 에이전트 실행 수명주기(Lifecycle) 훅을 가로채 장기 메모리를 인출하고 백그라운드 학습을 수행합니다.

1. **`before_agent` 훅**: 유저의 입력 메시지 쿼리를 기반으로 L2 에피소드 메모리를 자동 검색하고 L3 시멘틱 메모리를 프리페치하여 `AgentContext.recalled_memory`에 주입합니다.
2. **`after_agent` 훅 (Closed Learning Loop)**: 에이전트의 답변 턴이 끝난 직후 **비동기 데몬 스레드(Daemon Thread)**를 생성합니다. 별도의 LLM이 최근 대화 스냅샷을 검토(Background Review)하여 신규 팩트를 추출한 뒤 `memory(action='add', ...)` 도구를 자율 격발하여 `MEMORY.md`와 `USER.md`를 자동으로 업데이트합니다.

In [12]:
from app.utils.context import AgentContext
from modules.hermes.memory_middleware import MemoryMiddleware

# 메모리 옵션이 활성화된 AgentContext 정의
ctx = AgentContext(
    episodic_memory_enabled=True,
    semantic_memory_enabled=True,
    memory_learning_enabled=True,
    memory_dir=tmp_dir,
    episodic_db_path=os.path.join(tmp_dir, "episodic.db")
)

print("AgentContext Memory Configuration:")
print(f"- Episodic Memory Enabled: {ctx.episodic_memory_enabled}")
print(f"- Semantic Memory Enabled: {ctx.semantic_memory_enabled}")
print(f"- Closed Learning Loop Enabled: {ctx.memory_learning_enabled}")
print(f"- Recalled Memory Field: '{ctx.recalled_memory}'")

AgentContext Memory Configuration:
- Episodic Memory Enabled: True
- Semantic Memory Enabled: True
- Closed Learning Loop Enabled: True
- Recalled Memory Field: ''


## 5. End-to-End Frontier Agent 통합 구동 시연

`app/agents/frontier_agent.py`에 구축된 통합 에이전트를 생성하여 L1(AsyncSqliteSaver), L2(EpisodicStore), L3(SemanticMemoryStore) 및 도구들이 완전하게 연동되어 동작하는지 검증합니다.

In [13]:
your_name = "제이드" #여기 당신의 이름을 넣으시오. 

In [14]:
import asyncio
import time
from app.agents.frontier_agent import create_agent_executor

async def test_hermes_memory_retention_across_sessions():
    print("==========================================================================")
    print("🧪 [Hermes 4계층 메모리 세션 전환 & 기억 이관 테스트]")
    print("==========================================================================\n")

    # 1. 에이전트 팩토리 구동 (메인: gemini-3.5-flash, 리뷰: gpt-4o-mini)
    agent = await create_agent_executor()

    # --------------------------------------------------------------------------
    # 💬 [세션 1] 대화 진행: 유저 이름, 프로젝트 환경, 테스트 선호도 전달
    # --------------------------------------------------------------------------
    session1_config = {"configurable": {"thread_id": "session_001_initial"}}
    
    user_msg_1 = (
        f"안녕! 나는 파이썬 개발자 {your_name}라고 해. "
        "우리 프로젝트는 FastAPI와 PostgreSQL 15를 사용하고 있고, "
        "단위 테스트로는 sempre pytest를 우선적으로 선호해."
    )
    print("👤 [세션 1 유저]:", user_msg_1)
    
    input1 = {"messages": [("user", user_msg_1)]}
    res1 = await agent.ainvoke(input1, config=session1_config)
    print("\n🤖 [세션 1 에이전트 답변]:\n", res1["messages"][-1].content)
    
    print("\n⏳ [백그라운드 학습] gpt-4o-mini 데몬이 L1 대화를 관찰하고 L3 마크다운 팩트를 자동 추출 중...")
    await asyncio.sleep(3)  # 비동기 백그라운드 리뷰 완료 대기

    # --------------------------------------------------------------------------
    # 🔄 [세션 전환] 세션 2 시작 (새로운 thread_id = "session_002_new")
    # --------------------------------------------------------------------------
    print("\n" + "=" * 70)
    print("🔄 [세션 전환] 세션 1 종료 ➔ 세션 2 (session_002_new) 시작!")
    print("=" * 70 + "\n")

    session2_config = {"configurable": {"thread_id": "session_002_new"}}
    
    user_msg_2 = (
        "안녕! 내가 누구인지 기억나니? "
        "그리고 우리 프로젝트 기술 스택과 내가 선호하는 테스트 프레임워크가 무엇이었는지 알려줘."
    )
    print("👤 [세션 2 유저]:", user_msg_2)

    input2 = {"messages": [("user", user_msg_2)]}
    res2 = await agent.ainvoke(input2, config=session2_config)
    
    print("\n🤖 [세션 2 에이전트 답변 (기억 인출 기반)]:\n", res2["messages"][-1].content)
    print("\n🎉 세션 간 장기 기억 이관 및 인출 테스트 완료!")

await test_hermes_memory_retention_across_sessions()


🧪 [Hermes 4계층 메모리 세션 전환 & 기억 이관 테스트]

👤 [세션 1 유저]: 안녕! 나는 파이썬 개발자 제이드라고 해. 우리 프로젝트는 FastAPI와 PostgreSQL 15를 사용하고 있고, 단위 테스트로는 sempre pytest를 우선적으로 선호해.

🪵 [AgentTracer] === Agent Execution Started ===
📥 Query: 안녕! 나는 파이썬 개발자 제이드라고 해. 우리 프로젝트는 FastAPI와 PostgreSQL 15를 사용하고 있고, 단위 테스트로는 sempre pytest를 우선적으로 선호해.
🆔 Session: unknown
🧠 [AgentTracer (Async)] LLM response (7559 prompt / 47 completion tokens, took 2979ms)
🔧 [AgentTracer (Async)] Tool Executing: memory with args: {'content': 'User is Jade (제이드), a Python developer.', 'old_text': 'Park Hyun-woo', 'target': 'user', 'action': 'replace'}
🔧 [AgentTracer (Async)] Tool Executed in 0ms
🧠 [AgentTracer (Async)] LLM response (7806 prompt / 36 completion tokens, took 4747ms)
🔧 [AgentTracer (Async)] Tool Executing: memory with args: {'content': 'User is Jade (제이드), a Python developer.', 'target': 'user', 'action': 'add'}
🔧 [AgentTracer (Async)] Tool Executed in 4ms
🧠 [AgentTracer (Async)] LLM response (8388 prompt / 86 completion tokens, too

## 💡 요약 및 정리

1. **L1 Working Memory**: `AsyncSqliteSaver`를 통한 단기 세션 상태 보존.
2. **L2 Episodic Memory**: 세션 종료 후 요약 FTS5 인덱싱 및 Anchor 키워드 기반 주변 대화(Lineage) 인출.
3. **L3 Semantic Memory**: `§` 구분자 기반의 마크다운 팩트 관리 및 Frozen Snapshot 시스템 프롬프트 캐싱 보존.
4. **Closed Learning Loop**: `after_agent` 데몬 스레드에서 대화 자동 리뷰 및 `memory()` 도구 자율 격발을 통한 장기 메모리 연속 학습 구축 완료.